In [1]:
library(dplyr)
## visualization
library(ggplot2)
library(ggplot2)
library(gridExtra)
library(dplyr)
library(ggplot2)
library(cowplot)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




In [2]:
setwd("/home/liuxuan/metadata/PRJNA503603")
initial_meta<-read.csv("SraRunTable.csv",header = T)
head(initial_meta)

,Run,Assay.Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center.Name,Collection_Date,⋯,LibraryLayout,LibrarySelection,LibrarySource,Organism,Platform,ReleaseDate,create_date,version,Sample.Name,SRA.Study
,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>
1,SRR8148126,AMPLICON,594,27590133,PRJNA503603,SAMN10365967,Metagenome or environmental,14739559,"RENJI HOSPITAL, SCHOOL OF MEDICINE, SHANGHAI JIAO TONG UNIVERSITY",2018-10-15,⋯,PAIRED,PCR,METAGENOMIC,human saliva metagenome,ILLUMINA,2018-11-04T00:00:00Z,2018-11-04T02:57:00Z,1,E15,SRP167714
2,SRR8148127,AMPLICON,594,49667076,PRJNA503603,SAMN10365903,Metagenome or environmental,29360864,"RENJI HOSPITAL, SCHOOL OF MEDICINE, SHANGHAI JIAO TONG UNIVERSITY",2018-08-01,⋯,PAIRED,PCR,METAGENOMIC,human saliva metagenome,ILLUMINA,2018-11-04T00:00:00Z,2018-11-04T02:57:00Z,1,P9,SRP167714
3,SRR8148128,AMPLICON,594,45169960,PRJNA503603,SAMN10365950,Metagenome or environmental,27641582,"RENJI HOSPITAL, SCHOOL OF MEDICINE, SHANGHAI JIAO TONG UNIVERSITY",2018-08-01,⋯,PAIRED,PCR,METAGENOMIC,human saliva metagenome,ILLUMINA,2018-11-04T00:00:00Z,2018-11-04T02:57:00Z,1,N22,SRP167714
4,SRR8148129,AMPLICON,594,40345077,PRJNA503603,SAMN10365898,Metagenome or environmental,23934942,"RENJI HOSPITAL, SCHOOL OF MEDICINE, SHANGHAI JIAO TONG UNIVERSITY",2018-08-01,⋯,PAIRED,PCR,METAGENOMIC,human saliva metagenome,ILLUMINA,2018-11-04T00:00:00Z,2018-11-04T02:57:00Z,1,P4,SRP167714
5,SRR8148130,AMPLICON,594,40988962,PRJNA503603,SAMN10365897,Metagenome or environmental,24402409,"RENJI HOSPITAL, SCHOOL OF MEDICINE, SHANGHAI JIAO TONG UNIVERSITY",2018-08-01,⋯,PAIRED,PCR,METAGENOMIC,human saliva metagenome,ILLUMINA,2018-11-04T00:00:00Z,2018-11-04T02:57:00Z,1,P3,SRP167714
6,SRR8148131,AMPLICON,594,34488138,PRJNA503603,SAMN10365896,Metagenome or environmental,20714005,"RENJI HOSPITAL, SCHOOL OF MEDICINE, SHANGHAI JIAO TONG UNIVERSITY",2018-08-01,⋯,PAIRED,PCR,METAGENOMIC,human saliva metagenome,ILLUMINA,2018-11-04T00:00:00Z,2018-11-04T02:57:00Z,1,P2,SRP167714


In [3]:
colnames(initial_meta)  

[1] "Run"                            "Assay.Type"                    
 [3] "AvgSpotLen"                     "Bases"                         
 [5] "BioProject"                     "BioSample"                     
 [7] "BioSampleModel"                 "Bytes"                         
 [9] "Center.Name"                    "Collection_Date"               
[11] "Consent"                        "DATASTORE.filetype"            
[13] "DATASTORE.provider"             "DATASTORE.region"              
[15] "Experiment"                     "geo_loc_name_country"          
[17] "geo_loc_name_country_continent" "geo_loc_name"                  
[19] "Host"                           "Instrument"                    
[21] "isolation_source"               "lat_lon"                       
[23] "Library.Name"                   "LibraryLayout"                 
[25] "LibrarySelection"               "LibrarySource"                 
[27] "Organism"                       "Platform"                      
[29] "ReleaseDate"                    "create_date"                   
[31] "version"                        "Sample.Name"                   
[33] "SRA.Study"

In [4]:
uniform_meta <- data.frame(
    BioProject = initial_meta$BioProject,
    Run = initial_meta$Run,
    sample_name = initial_meta$'Sample.Name',
    Age = NA,
    BMI = NA,
    Sex = NA,
    Region = "V3-V4",
    Platform = initial_meta$Platform,
    Country = initial_meta$geo_loc_name_country,
    Site = 'saliva',
    public_time = initial_meta$create_date,
    PCR_primers = 'F: ACTCCTACGGGAGGCAGCA; R: GGACTACHVGGGTWTCTAAT',
    title = 'Analysis of by high-throughput sequencing: Helicobacter pylori infection and salivary microbiome',
    Disease_study = "YES",
    Status = NA,
    matched_samples = 'NO'
)

In [5]:
# 根据样本名称设置状态
for (i in 1:nrow(uniform_meta)) {
    sample_name <- uniform_meta$sample_name[i]
    
    if (grepl("E", sample_name, ignore.case = TRUE)) {
        uniform_meta$Status[i] <- "eradicated group"
    } else if (grepl("P", sample_name, ignore.case = TRUE)) {
        uniform_meta$Status[i] <- "infected group"
    } else if (grepl("N", sample_name, ignore.case = TRUE)) {
        uniform_meta$Status[i] <- "uninfected group"
    }
    # 如果不匹配任何模式，保持为NA
}

In [6]:
head(uniform_meta)

,BioProject,Run,sample_name,Age,BMI,Sex,Region,Platform,Country,Site,public_time,PCR_primers,title,Disease_study,Status,matched_samples
,<chr>,<chr>,<chr>,<lgl>,<lgl>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,PRJNA503603,SRR8148126,E15,NA,NA,NA,V3-V4,ILLUMINA,China,saliva,2018-11-04T02:57:00Z,F: ACTCCTACGGGAGGCAGCA; R: GGACTACHVGGGTWTCTAAT,Analysis of by high-throughput sequencing: Helicobacter pylori infection and salivary microbiome,YES,eradicated group,NO
2,PRJNA503603,SRR8148127,P9,NA,NA,NA,V3-V4,ILLUMINA,China,saliva,2018-11-04T02:57:00Z,F: ACTCCTACGGGAGGCAGCA; R: GGACTACHVGGGTWTCTAAT,Analysis of by high-throughput sequencing: Helicobacter pylori infection and salivary microbiome,YES,infected group,NO
3,PRJNA503603,SRR8148128,N22,NA,NA,NA,V3-V4,ILLUMINA,China,saliva,2018-11-04T02:57:00Z,F: ACTCCTACGGGAGGCAGCA; R: GGACTACHVGGGTWTCTAAT,Analysis of by high-throughput sequencing: Helicobacter pylori infection and salivary microbiome,YES,uninfected group,NO
4,PRJNA503603,SRR8148129,P4,NA,NA,NA,V3-V4,ILLUMINA,China,saliva,2018-11-04T02:57:00Z,F: ACTCCTACGGGAGGCAGCA; R: GGACTACHVGGGTWTCTAAT,Analysis of by high-throughput sequencing: Helicobacter pylori infection and salivary microbiome,YES,infected group,NO
5,PRJNA503603,SRR8148130,P3,NA,NA,NA,V3-V4,ILLUMINA,China,saliva,2018-11-04T02:57:00Z,F: ACTCCTACGGGAGGCAGCA; R: GGACTACHVGGGTWTCTAAT,Analysis of by high-throughput sequencing: Helicobacter pylori infection and salivary microbiome,YES,infected group,NO
6,PRJNA503603,SRR8148131,P2,NA,NA,NA,V3-V4,ILLUMINA,China,saliva,2018-11-04T02:57:00Z,F: ACTCCTACGGGAGGCAGCA; R: GGACTACHVGGGTWTCTAAT,Analysis of by high-throughput sequencing: Helicobacter pylori infection and salivary microbiome,YES,infected group,NO


In [7]:
print(length(unique(uniform_meta$Run)))

print(length(unique(uniform_meta$sample_name)))

[1] 80
[1] 80


In [8]:
plot_data <- uniform_meta %>%
  select(-BioProject, -Run, -sample_name, -title)

In [9]:
create_simple_plots <- function(data) {
  plots <- list()
  
  for (col in colnames(data)) {
    # Check if the column is all NA
    if (all(is.na(data[[col]]))) {
      # Create an empty plot with text indicating all NA
      p <- ggplot() +
        annotate("text", x = 0.5, y = 0.5, label = "this column is NA", size = 6) +
        labs(title = col) +
        theme_void()
    } else if (is.numeric(data[[col]])) {
      # Numeric variable: histogram
      p <- ggplot(data, aes(x = .data[[col]])) +
        geom_histogram(fill = "blue", alpha = 0.6, bins = 20, na.rm = TRUE) +
        labs(title = col) +
        theme_minimal()
    } else {
      # Categorical variable: bar plot (remove NA values)
      count_data <- as.data.frame(table(data[[col]], useNA = "ifany"))
      p <- ggplot(count_data, aes(x = reorder(Var1, -Freq), y = Freq)) +
        geom_col(fill = "red", alpha = 0.6) +
        labs(title = col, x = NULL, y = "Count") +
        theme_minimal() +
        theme(axis.text.x = element_text(angle = 15, hjust = 1))
    }
    plots[[col]] <- p
  }
  
  return(plots)
}

In [10]:
plots <- create_simple_plots(plot_data)
p<-plot_grid(plots$Age, 
          plots$BMI, 
          plots$Sex, 
          plots$Region, 
          plots$Platform, 
          plots$Country, 
          plots$Site,
          plots$public_time, 
          plots$PCR_primers,
          plots$Disease_study,
          plots$Status,
          plots$matched_samples,# add 
          ncol = 3, align = "hv", axis = "lb")
ggsave("Metadata_Summary.png", plot = p, width = 8, height = 12, dpi = 150)

In [11]:
## uniform the column order
uniform_meta <- uniform_meta[, c("BioProject", "Run", "sample_name", "Age", "BMI", "Sex", "Region", "Platform", "Country", "Site", "public_time", "PCR_primers", "title", "Disease_study", "Status")]
write.csv(uniform_meta, file = "/home/liuxuan/metadata/PRJNA503603/uniform_meta.csv", quote = F)